<div style="background: linear-gradient(135deg, #000000 0%, #0a0a0a 50%, #1a0000 100%);
            padding: 30px 40px;
            border-radius: 4px;
            text-align: center;
            border-left: 4px solid #ff5347;
            border-right: 4px solid #ff5347;
            box-shadow: 0 0 40px rgba(255, 83, 71, 0.3), inset 0 0 60px rgba(255, 83, 71, 0.05);">
    <h1 style="color: #fafafa;
               font-family: monospace;
               font-size: 3.5em;
               letter-spacing: 0.5em;
               text-shadow: 0 0 20px rgba(255, 83, 71, 0.9), 0 0 40px rgba(255, 83, 71, 0.4);
               margin: 0;
               padding: 10px 0;">
        ML-KEM
    </h1>
    <div style="margin-top: 15px; color: #ff8a80; font-family: monospace; font-size: 0.75em; letter-spacing: 0.3em;">
        Michal Kowaslki
    </div>
</div>

In [1]:
import os

# Use the locally built liboqs install if available
oqs_install_path = r"C:\Users\Admin\_oqs_0.15.0"
if os.path.isdir(oqs_install_path):
    os.environ["OQS_INSTALL_PATH"] = oqs_install_path
    print("Using OQS_INSTALL_PATH:", os.environ["OQS_INSTALL_PATH"])

import oqs
import time
import numpy as np
import matplotlib.pyplot as plt

Using OQS_INSTALL_PATH: C:\Users\Admin\_oqs_0.15.0


c:\Users\Admin\Desktop\benchmark project\Benchmarking_Post-Quantum_Cryptographic_Algorithms\.venv\Lib\site-packages\oqs\__init__.py:1: UserWarning: liboqs version (major, minor) 0.15.0 differs from liboqs-python version 0.14.1
  from oqs.oqs import (


In [2]:
import csv
import tracemalloc
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding, rsa
from cryptography.hazmat.primitives.serialization import Encoding, NoEncryption, PrivateFormat, PublicFormat

benchmark_output_path = "ml_kem_benchmark_results.txt"
csv_output_path = "ml_kem_timing.csv"
rsa_benchmark_output_path = "ecdh_p256_benchmark_results.txt"
rsa_csv_output_path = "ecdh_p256_timing.csv"


def write_header(file_path, header_text):
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(header_text + "\n")
        f.write("=" * len(header_text) + "\n")

write_header(benchmark_output_path, "ML-KEM Benchmark Results")
write_header(rsa_benchmark_output_path, "ECDH-P-256 Benchmark Results")

csv_enabled = True
try:
    with open(csv_output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(["metric", "run", "time_ms", "peak_kb"])
    with open(rsa_csv_output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(["metric", "run", "time_ms", "peak_kb"])
except PermissionError:
    print(f"Warning: Cannot write to one or more CSV files (file may be open in another program). CSV logging disabled.")
    csv_enabled = False




def append_to_output(file_path, *lines):
    with open(file_path, "a", encoding="utf-8") as f:
        for line in lines:
            f.write(str(line) + "\n")


def append_to_csv(csv_path, metric, run, time_ms, peak_kb):
    if csv_enabled:
        with open(csv_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f, delimiter=';')
            writer.writerow([metric, run, time_ms, peak_kb])

In [3]:
# Check available ML-KEM variants
print("Available ML-KEM algorithms:")
for alg in oqs.get_enabled_kem_mechanisms():
    if "ML-KEM" in alg:
        print(f"  - {alg}")

Available ML-KEM algorithms:
  - ML-KEM-512
  - ML-KEM-768
  - ML-KEM-1024


In [4]:
# Benchmark ML-KEM keypair generation for a representative parameter set
alg = "ML-KEM-768"
if alg not in oqs.get_enabled_kem_mechanisms():
    raise ValueError(f"Algorithm {alg} is not enabled. Choose one of: {[a for a in oqs.get_enabled_kem_mechanisms() if 'ML-KEM' in a]}")

print(f"Benchmarking {alg} keypair generation...")
runs = 20
times_ms = []
peaks_kb = []
with oqs.KeyEncapsulation(alg) as kem:
    for run_index in range(runs):
        start = time.perf_counter()
        tracemalloc.start()
        _ = kem.generate_keypair()
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        end = time.perf_counter()
        elapsed_ms = (end - start) * 1000
        peak_kb = peak / 1024
        times_ms.append(elapsed_ms)
        peaks_kb.append(peak_kb)
        append_to_csv(csv_output_path, "keypair_generation", run_index + 1, elapsed_ms, peak_kb)

print(f"Algorithm: {alg}")
print(f"Runs: {runs}")
print(f"Mean keypair time: {np.mean(times_ms):.3f} ms")
print(f"Min keypair time: {np.min(times_ms):.3f} ms")
print(f"Max keypair time: {np.mean(times_ms):.3f} ms")
print(f"Mean peak memory: {np.mean(peaks_kb):.3f} KB")
print(f"Max peak memory: {np.max(peaks_kb):.3f} KB")
print("Times (ms):", [round(t, 3) for t in times_ms])
print("Peaks (KB):", [round(p, 3) for p in peaks_kb])

append_to_output(
    benchmark_output_path,
    f"\nBenchmarking {alg} keypair generation...",
    f"Algorithm: {alg}",
    f"Runs: {runs}",
    f"Mean keypair time: {np.mean(times_ms):.3f} ms",
    f"Min keypair time: {np.min(times_ms):.3f} ms",
    f"Max keypair time: {np.max(times_ms):.3f} ms",
    f"Mean peak memory: {np.mean(peaks_kb):.3f} KB",
    f"Max peak memory: {np.max(peaks_kb):.3f} KB",
    f"Times (ms): {[round(t, 3) for t in times_ms]}",
    f"Peaks (KB): {[round(p, 3) for p in peaks_kb]}")

Benchmarking ML-KEM-768 keypair generation...
Algorithm: ML-KEM-768
Runs: 20
Mean keypair time: 0.539 ms
Min keypair time: 0.356 ms
Max keypair time: 0.539 ms
Mean peak memory: 5.265 KB
Max peak memory: 11.284 KB
Times (ms): [0.752, 0.686, 0.601, 0.723, 0.386, 0.591, 0.372, 0.498, 0.356, 0.375, 0.516, 0.523, 0.734, 0.852, 0.559, 0.565, 0.423, 0.517, 0.382, 0.372]
Peaks (KB): [11.284, 4.923, 4.923, 4.923, 4.923, 4.923, 4.923, 4.923, 4.923, 5.41, 4.923, 4.923, 4.923, 4.923, 4.923, 4.923, 4.923, 4.923, 4.923, 4.923]


In [5]:
# Benchmark ML-KEM encapsulation
print(f"Benchmarking {alg} encapsulation...")
runs = 20
encap_times_ms = []
encap_peaks_kb = []
with oqs.KeyEncapsulation(alg) as kem:
    public_key = kem.generate_keypair()
    for run_index in range(runs):
        start = time.perf_counter()
        tracemalloc.start()
        ciphertext, shared_secret_enc = kem.encap_secret(public_key)
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        end = time.perf_counter()
        elapsed_ms = (end - start) * 1000
        peak_kb = peak / 1024
        encap_times_ms.append(elapsed_ms)
        encap_peaks_kb.append(peak_kb)
        append_to_csv(csv_output_path, "encapsulation", run_index + 1, elapsed_ms, peak_kb)

print(f"Algorithm: {alg}")
print(f"Runs: {runs}")
print(f"Mean encapsulation time: {np.mean(encap_times_ms):.3f} ms")
print(f"Min encapsulation time: {np.min(encap_times_ms):.3f} ms")
print(f"Max encapsulation time: {np.max(encap_times_ms):.3f} ms")
print(f"Mean peak memory: {np.mean(encap_peaks_kb):.3f} KB")
print(f"Max peak memory: {np.max(encap_peaks_kb):.3f} KB")
print("Encapsulation times (ms):", [round(t, 3) for t in encap_times_ms])
print("Peaks (KB):", [round(p, 3) for p in encap_peaks_kb])

append_to_output(
    benchmark_output_path,
    f"\nBenchmarking {alg} encapsulation...",
    f"Algorithm: {alg}",
    f"Runs: {runs}",
    f"Mean encapsulation time: {np.mean(encap_times_ms):.3f} ms",
    f"Min encapsulation time: {np.min(encap_times_ms):.3f} ms",
    f"Max encapsulation time: {np.max(encap_times_ms):.3f} ms",
    f"Mean peak memory: {np.mean(encap_peaks_kb):.3f} KB",
    f"Max peak memory: {np.max(encap_peaks_kb):.3f} KB",
    f"Encapsulation times (ms): {[round(t, 3) for t in encap_times_ms]}",
    f"Peaks (KB): {[round(p, 3) for p in encap_peaks_kb]}")

Benchmarking ML-KEM-768 encapsulation...
Algorithm: ML-KEM-768
Runs: 20
Mean encapsulation time: 0.468 ms
Min encapsulation time: 0.379 ms
Max encapsulation time: 0.809 ms
Mean peak memory: 4.075 KB
Max peak memory: 10.064 KB
Encapsulation times (ms): [0.797, 0.476, 0.386, 0.405, 0.389, 0.4, 0.383, 0.379, 0.39, 0.522, 0.567, 0.809, 0.401, 0.412, 0.382, 0.476, 0.381, 0.397, 0.384, 0.614]
Peaks (KB): [10.064, 3.76, 3.76, 3.76, 3.76, 3.76, 3.76, 3.76, 3.76, 3.76, 3.76, 3.76, 3.76, 3.76, 3.76, 3.76, 3.76, 3.76, 3.76, 3.76]


In [6]:
# Benchmark ML-KEM decapsulation
print(f"Benchmarking {alg} decapsulation...")
runs = 20
decap_times_ms = []
decap_peaks_kb = []
with oqs.KeyEncapsulation(alg) as kem:
    public_key = kem.generate_keypair()
    ciphertext, shared_secret_enc = kem.encap_secret(public_key)
    for run_index in range(runs):
        start = time.perf_counter()
        tracemalloc.start()
        shared_secret_dec = kem.decap_secret(ciphertext)
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        end = time.perf_counter()
        elapsed_ms = (end - start) * 1000
        peak_kb = peak / 1024
        decap_times_ms.append(elapsed_ms)
        decap_peaks_kb.append(peak_kb)
        append_to_csv(csv_output_path, "decapsulation", run_index + 1, elapsed_ms, peak_kb)

print(f"Algorithm: {alg}")
print(f"Runs: {runs}")
print(f"Mean decapsulation time: {np.mean(decap_times_ms):.3f} ms")
print(f"Min decapsulation time: {np.min(decap_times_ms):.3f} ms")
print(f"Max decapsulation time: {np.max(decap_times_ms):.3f} ms")
print(f"Mean peak memory: {np.mean(decap_peaks_kb):.3f} KB")
print(f"Max peak memory: {np.max(decap_peaks_kb):.3f} KB")
print("Decapsulation times (ms):", [round(t, 3) for t in decap_times_ms])
print("Peaks (KB):", [round(p, 3) for p in decap_peaks_kb])

append_to_output(
    benchmark_output_path,
    f"\nBenchmarking {alg} decapsulation...",
    f"Algorithm: {alg}",
    f"Runs: {runs}",
    f"Mean decapsulation time: {np.mean(decap_times_ms):.3f} ms",
    f"Min decapsulation time: {np.min(decap_times_ms):.3f} ms",
    f"Max decapsulation time: {np.max(decap_times_ms):.3f} ms",
    f"Mean peak memory: {np.mean(decap_peaks_kb):.3f} KB",
    f"Max peak memory: {np.max(decap_peaks_kb):.3f} KB",
    f"Decapsulation times (ms): {[round(t, 3) for t in decap_times_ms]}",
    f"Peaks (KB): {[round(p, 3) for p in decap_peaks_kb]}")

Benchmarking ML-KEM-768 decapsulation...
Algorithm: ML-KEM-768
Runs: 20
Mean decapsulation time: 0.540 ms
Min decapsulation time: 0.446 ms
Max decapsulation time: 0.752 ms
Mean peak memory: 1.668 KB
Max peak memory: 2.276 KB
Decapsulation times (ms): [0.515, 0.56, 0.513, 0.741, 0.532, 0.446, 0.464, 0.454, 0.716, 0.752, 0.482, 0.467, 0.482, 0.489, 0.506, 0.639, 0.476, 0.497, 0.599, 0.476]
Peaks (KB): [2.117, 1.609, 1.609, 1.609, 1.609, 1.609, 1.609, 1.609, 1.609, 1.609, 1.609, 1.609, 2.276, 1.609, 1.609, 1.609, 1.609, 1.609, 1.609, 1.609]


In [7]:
# Measure ML-KEM size metrics
print(f"Measuring {alg} size metrics...")
with oqs.KeyEncapsulation(alg) as kem:
    public_key = kem.generate_keypair()
    secret_key = kem.export_secret_key()
    ciphertext, shared_secret = kem.encap_secret(public_key)

print(f"Algorithm: {alg}")
print(f"Public key size: {len(public_key)} bytes")
print(f"Secret key size: {len(secret_key)} bytes")
print(f"Ciphertext size: {len(ciphertext)} bytes")
print(f"Shared secret size: {len(shared_secret)} bytes")

append_to_output(
    benchmark_output_path,
    f"\nMeasuring {alg} size metrics...",
    f"Algorithm: {alg}",
    f"Public key size: {len(public_key)} bytes",
    f"Secret key size: {len(secret_key)} bytes",
    f"Ciphertext size: {len(ciphertext)} bytes",
    f"Shared secret size: {len(shared_secret)} bytes")

Measuring ML-KEM-768 size metrics...
Algorithm: ML-KEM-768
Public key size: 1184 bytes
Secret key size: 2400 bytes
Ciphertext size: 1088 bytes
Shared secret size: 32 bytes


In [8]:
# ECDH-P-256 comparison benchmark
from cryptography.hazmat.primitives.asymmetric import ec
ecdh_runs = 20

print("Benchmarking ECDH-P-256 keypair generation...")
ecdh_keygen_times = []
ecdh_keygen_peaks_kb = []
for run_index in range(ecdh_runs):
    start = time.perf_counter()
    tracemalloc.start()
    private_key = ec.generate_private_key(ec.SECP256R1())
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    end = time.perf_counter()
    elapsed_ms = (end - start) * 1000
    peak_kb = peak / 1024
    ecdh_keygen_times.append(elapsed_ms)
    ecdh_keygen_peaks_kb.append(peak_kb)
    append_to_csv(rsa_csv_output_path, "keypair_generation", run_index + 1, elapsed_ms, peak_kb)

print(f"Runs: {ecdh_runs}")
print(f"Mean keypair time: {np.mean(ecdh_keygen_times):.3f} ms")
print(f"Min keypair time: {np.min(ecdh_keygen_times):.3f} ms")
print(f"Max keypair time: {np.max(ecdh_keygen_times):.3f} ms")
print(f"Mean peak memory: {np.mean(ecdh_keygen_peaks_kb):.3f} KB")
print(f"Max peak memory: {np.max(ecdh_keygen_peaks_kb):.3f} KB")
print("Times (ms):", [round(t, 3) for t in ecdh_keygen_times])
print("Peaks (KB):", [round(p, 3) for p in ecdh_keygen_peaks_kb])

append_to_output(
    rsa_benchmark_output_path,
    "\nBenchmarking ECDH-P-256 keypair generation...",
    f"Runs: {ecdh_runs}",
    f"Mean keypair time: {np.mean(ecdh_keygen_times):.3f} ms",
    f"Min keypair time: {np.min(ecdh_keygen_times):.3f} ms",
    f"Max keypair time: {np.max(ecdh_keygen_times):.3f} ms",
    f"Mean peak memory: {np.mean(ecdh_keygen_peaks_kb):.3f} KB",
    f"Max peak memory: {np.max(ecdh_keygen_peaks_kb):.3f} KB",
    f"Times (ms): {[round(t, 3) for t in ecdh_keygen_times]}",
    f"Peaks (KB): {[round(p, 3) for p in ecdh_keygen_peaks_kb]}")

print("Benchmarking ECDH-P-256 shared secret derivation...")
# Generate a fixed peer public key for consistent benchmarking
peer_private = ec.generate_private_key(ec.SECP256R1())
peer_public = peer_private.public_key()

ecdh_derive_times = []
ecdh_derive_peaks_kb = []
for run_index in range(ecdh_runs):
    # Generate new private key each time
    private_key = ec.generate_private_key(ec.SECP256R1())
    start = time.perf_counter()
    tracemalloc.start()
    shared_secret = private_key.exchange(ec.ECDH(), peer_public)
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    end = time.perf_counter()
    elapsed_ms = (end - start) * 1000
    peak_kb = peak / 1024
    ecdh_derive_times.append(elapsed_ms)
    ecdh_derive_peaks_kb.append(peak_kb)
    append_to_csv(rsa_csv_output_path, "shared_secret_derivation", run_index + 1, elapsed_ms, peak_kb)

print(f"Runs: {ecdh_runs}")
print(f"Mean derivation time: {np.mean(ecdh_derive_times):.3f} ms")
print(f"Min derivation time: {np.min(ecdh_derive_times):.3f} ms")
print(f"Max derivation time: {np.max(ecdh_derive_times):.3f} ms")
print(f"Mean peak memory: {np.mean(ecdh_derive_peaks_kb):.3f} KB")
print(f"Max peak memory: {np.max(ecdh_derive_peaks_kb):.3f} KB")
print("Derivation times (ms):", [round(t, 3) for t in ecdh_derive_times])
print("Peaks (KB):", [round(p, 3) for p in ecdh_derive_peaks_kb])

append_to_output(
    rsa_benchmark_output_path,
    "\nBenchmarking ECDH-P-256 shared secret derivation...",
    f"Runs: {ecdh_runs}",
    f"Mean derivation time: {np.mean(ecdh_derive_times):.3f} ms",
    f"Min derivation time: {np.min(ecdh_derive_times):.3f} ms",
    f"Max derivation time: {np.max(ecdh_derive_times):.3f} ms",
    f"Mean peak memory: {np.mean(ecdh_derive_peaks_kb):.3f} KB",
    f"Max peak memory: {np.max(ecdh_derive_peaks_kb):.3f} KB",
    f"Derivation times (ms): {[round(t, 3) for t in ecdh_derive_times]}",
    f"Peaks (KB): {[round(p, 3) for p in ecdh_derive_peaks_kb]}")

print("Measuring ECDH-P-256 size metrics...")
# Use the last generated keys
private_bytes = private_key.private_bytes(
    encoding=Encoding.PEM,
    format=PrivateFormat.PKCS8,
    encryption_algorithm=NoEncryption(),
)
public_bytes = private_key.public_key().public_bytes(
    encoding=Encoding.PEM,
    format=PublicFormat.SubjectPublicKeyInfo,
)

print(f"Public key size: {len(public_bytes)} bytes")
print(f"Private key size: {len(private_bytes)} bytes")
print(f"Shared secret size: {len(shared_secret)} bytes")

append_to_output(
    rsa_benchmark_output_path,
    "\nMeasuring ECDH-P-256 size metrics...",
    f"Public key size: {len(public_bytes)} bytes",
    f"Private key size: {len(private_bytes)} bytes",
    f"Shared secret size: {len(shared_secret)} bytes")

Benchmarking ECDH-P-256 keypair generation...
Runs: 20
Mean keypair time: 0.354 ms
Min keypair time: 0.080 ms
Max keypair time: 4.928 ms
Mean peak memory: 0.308 KB
Max peak memory: 1.060 KB
Times (ms): [4.928, 0.183, 0.266, 0.113, 0.12, 0.102, 0.127, 0.095, 0.117, 0.087, 0.08, 0.092, 0.087, 0.081, 0.081, 0.102, 0.084, 0.107, 0.131, 0.097]
Peaks (KB): [1.06, 0.336, 0.328, 0.32, 0.305, 0.297, 0.289, 0.281, 0.273, 0.266, 0.258, 0.25, 0.236, 0.236, 0.236, 0.236, 0.236, 0.236, 0.236, 0.236]
Benchmarking ECDH-P-256 shared secret derivation...
Runs: 20
Mean derivation time: 0.153 ms
Min derivation time: 0.119 ms
Max derivation time: 0.602 ms
Mean peak memory: 0.302 KB
Max peak memory: 0.585 KB
Derivation times (ms): [0.602, 0.138, 0.129, 0.129, 0.145, 0.125, 0.124, 0.122, 0.123, 0.124, 0.121, 0.122, 0.124, 0.12, 0.119, 0.123, 0.136, 0.128, 0.127, 0.183]
Peaks (KB): [0.585, 0.36, 0.353, 0.345, 0.337, 0.329, 0.321, 0.306, 0.298, 0.29, 0.282, 0.274, 0.267, 0.259, 0.251, 0.236, 0.236, 0.236, 0.23